In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report 


#----------------------------------------------------------------------------------------------------------------------------------#
#                                                           特征工程                                                                #
#----------------------------------------------------------------------------------------------------------------------------------#

## 数据导入
train_data = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
test_data = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")

## 家庭大小 
train_data['FamilySize'] = train_data['SibSp'] + train_data['Parch'] + 1
test_data['FamilySize'] = test_data['SibSp'] + test_data['Parch'] + 1
train_data['IsAlone'] = (train_data['FamilySize'] == 1).astype(int)
test_data['IsAlone'] = (test_data['FamilySize'] == 1).astype(int)

features = ["Pclass", "Sex", "Age", "FamilySize","IsAlone"]
target = "Survived"

x_full = train_data[features]
y_full = train_data[target]

x_train, x_val, y_train, y_val = train_test_split(x_full, y_full,test_size=0.2,random_state=78,stratify=y_full)

#-------------------------------------------------------------------------------------------------------------------------------------#
#                                                              数据处理                                                                #
#-------------------------------------------------------------------------------------------------------------------------------------#
## 缺失值处理
x_train = x_train.copy()
x_val = x_val.copy()
x_full = x_full.copy()
test_data = test_data.copy()

age_median = x_full['Age'].median()
fare_median = x_full['Fare'].median() if 'Fare' in features else None

x_train['Age'] = x_train['Age'].fillna(age_median)
x_val['Age'] =  x_val['Age'].fillna(age_median)
test_data['Age'] = test_data['Age'].fillna(age_median)

if fare_median is not None : 
    x_train['Fare'] = x_train['Fare'].fillna(fare_median)
    x_val['Fare'] = x_val['Fare'].fillna(fare_median)
    test_data['Fare'] = test_data['Fare'].fillna(fare_median)

## 分箱
bins = [0, 12, 18, 60, 100]
labels = [0, 1, 2, 3]
for df in [x_train, x_val, x_full, test_data]:
    df['Age_Bin'] = pd.cut(df['Age'], bins=bins, labels=labels, include_lowest=True)

#---------------------------------------------------------------------------------------------------------------------------------------#
#                                                                                                                                       #
#---------------------------------------------------------------------------------------------------------------------------------------#
##维度轴数+维度对齐
final_features = ["Pclass", "Sex", "Age_Bin", "FamilySize", "IsAlone"]

x_train_encoded = pd.get_dummies(x_train[final_features], drop_first=False)
x_val_encoded = pd.get_dummies(x_val[final_features], drop_first=False)
x_full_encoded = pd.get_dummies(x_full[final_features], drop_first=False)
x_test_encoded = pd.get_dummies(test_data[final_features], drop_first=False)

x_val_encoded = x_val_encoded.reindex(columns=x_train_encoded.columns, fill_value=0)
x_full_encoded = x_full_encoded.reindex(columns=x_train_encoded.columns, fill_value=0)
x_test_encoded = x_test_encoded.reindex(columns=x_train_encoded.columns, fill_value=0)


##模型设置
submit = True

if not submit:
    model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=91)
    model.fit(x_train_encoded, y_train)    
    val_predictions = model.predict(x_val_encoded)
    print(f"本地验证准确率: {accuracy_score(y_val, val_predictions):.4f}")  
    print(classification_report(y_val, val_predictions))                      

else:
    model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=91)
    model.fit(x_full_encoded, y_full)                            
    test_predictions = model.predict(x_test_encoded)                          
    
    ## 输出
    submission = pd.DataFrame({"PassengerId": test_data["PassengerId"], "Survived": test_predictions})
    submission.to_csv("submission.csv", index=False)
    